# Notebook 5 - RAGAS Evaluation
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

Evaluates the RAG pipeline using **RAGAS** (Retrieval-Augmented Generation Assessment Suite).
Uses your local Ollama LLM as the judge, so no API keys are needed and no data leaves your machine.

| Metric | What it measures | Range |
|---|---|---|
| **Faithfulness** | Does the answer stick to retrieved context? (no hallucination) | 0-1 (higher better) |
| **Answer Relevancy** | Does the answer actually address the question? | 0-1 (higher better) |
| **Context Precision** | Are retrieved chunks relevant to the question? | 0-1 (higher better) |
| **Context Recall** | Do retrieved chunks contain everything needed for the ground-truth answer? | 0-1 (higher better) |

**Requirements:** Notebooks 1-3 complete, Ollama running with `llama3.1:8b` (judge) and `intfloat/multilingual-e5-base` for embeddings.

## Cell 1 - Install RAGAS (one-time)

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# RAGAS + langchain integrations needed for Ollama judge
pip("ragas>=0.2.0", "langchain-ollama", "langchain-huggingface", "datasets", "nltk")
print("[ok] RAGAS installed")


## Cell 2 - Imports, config, FAISS reload

In [ ]:
import os, json, time
from pathlib import Path
import pandas as pd
import numpy as np
import faiss

DRIVE_BASE = Path(os.environ.get("RAG_UPI_BASE", r"D:\Project\RAG_UPI\Dataset"))
PIPE = DRIVE_BASE / "_pipeline"
INDEX_FILE = PIPE / "index" / "faiss.index"
META_FILE  = PIPE / "index" / "chunks_meta.json"
EVAL_DIR   = PIPE / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "embedding_model": "intfloat/multilingual-e5-base",
    "use_e5_prefixes": True,
    "judge_model": "llama3.1:8b",        # Ollama model used by RAGAS as judge
    "ollama_base_url": "http://localhost:11434",
    "top_k": 5,
    "max_eval_questions": 30,            # cap for runtime; bump if you have time
    "csv_candidates": [
        DRIVE_BASE / "evaluation.csv",
    ],
}

# --- Load FAISS ---
index = faiss.read_index(str(INDEX_FILE))
meta  = json.loads(META_FILE.read_text(encoding="utf-8"))
print(f"FAISS loaded: {index.ntotal} vectors x dim={index.d}")

# --- Embedding model (same as N3) ---
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(CONFIG["embedding_model"])

def embed(texts, kind="passage"):
    if CONFIG["use_e5_prefixes"]:
        tag = "query: " if kind == "query" else "passage: "
        texts = [tag + t for t in texts]
    v = embedder.encode(texts, batch_size=32, convert_to_numpy=True,
                        normalize_embeddings=True).astype("float32")
    return v

def retrieve(query, k=None):
    k = k or CONFIG["top_k"]
    q = embed([query], kind="query")
    scores, idxs = index.search(q, k)
    return [{**meta[i], "score": float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]

print(f"Embedder dim: {embedder.get_sentence_embedding_dimension()}")


## Cell 3 - Load evaluation queries

In [ ]:
def load_eval_dataset():
    """Build a HuggingFace Dataset shape: question, contexts, answer, ground_truth.

    Pulls (query, context) pairs from evaluation.csv. Generates answers by
    calling the SAME retrieval + LLM pipeline you ship in production.
    """
    csv_path = next((p for p in CONFIG["csv_candidates"] if p.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "No evaluation.csv found. Put one at " + str(CONFIG["csv_candidates"][0])
        )
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from {csv_path.name}")
    df = df.dropna(subset=["query"]).head(CONFIG["max_eval_questions"])
    return df

eval_df = load_eval_dataset()
print(eval_df[["query"]].head())


## Cell 4 - Run RAG on each eval query to collect answers + contexts

In [ ]:
import requests

def ollama_generate(prompt, model=None, temperature=0.1):
    r = requests.post(
        f"{CONFIG['ollama_base_url']}/api/generate",
        json={
            "model": model or CONFIG["judge_model"],
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature, "num_predict": 1024},
        },
        timeout=240,
    )
    r.raise_for_status()
    return (r.json().get("response") or "").strip()


def grounded_answer(query, retrieved):
    """Same prompt shape as the production backend (numbered citations)."""
    context = "\n\n".join(
        f"[{i+1}] {c['title']} (hal. {c['page']})\n{c['text']}"
        for i, c in enumerate(retrieved)
    )
    prompt = f"""Anda adalah asisten resmi UPI. Jawab HANYA dari SUMBER bernomor.
Setiap fakta wajib diikuti [1], [2] dst. Jangan mengarang. Bahasa Indonesia baku.

SUMBER:
{context}

PERTANYAAN: {query}

JAWABAN (sertakan [n]):"""
    return ollama_generate(prompt)


records = []
for i, row in eval_df.iterrows():
    q = row["query"]
    print(f"[{i+1}/{len(eval_df)}] {q[:80]}")
    t0 = time.time()
    hits = retrieve(q, CONFIG["top_k"])
    contexts = [h["text"] for h in hits]
    answer = grounded_answer(q, hits)
    dt = time.time() - t0
    records.append({
        "question": q,
        "answer": answer,
        "contexts": contexts,
        # If your CSV has a `context` column, it serves as ground-truth reference.
        "ground_truth": (row.get("context") or "").strip() or None,
        "latency_s": round(dt, 2),
    })
    print(f"   -> {dt:.1f}s")

print(f"\nCollected {len(records)} eval records")


## Cell 5 - Score with RAGAS

In [ ]:
from datasets import Dataset
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    Faithfulness, AnswerRelevancy,
    ContextPrecision, ContextRecall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Local LLM judge through langchain-ollama
judge_llm = ChatOllama(
    model=CONFIG["judge_model"],
    base_url=CONFIG["ollama_base_url"],
    temperature=0.0,
    num_predict=512,
)
judge_emb = OllamaEmbeddings(
    model="nomic-embed-text",   # small, fast; pull with: ollama pull nomic-embed-text
    base_url=CONFIG["ollama_base_url"],
)

# Wrap for RAGAS
ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(judge_emb)

# Filter to records that have a ground_truth (needed for Context Recall).
records_with_gt = [r for r in records if r["ground_truth"]]
records_for_general = records   # all records can score Faithfulness / AnswerRelevancy

ds_general = Dataset.from_list([
    {"question": r["question"], "answer": r["answer"], "contexts": r["contexts"]}
    for r in records_for_general
])
print(f"Scoring Faithfulness + AnswerRelevancy on {len(ds_general)} records ...")
result_general = evaluate(
    ds_general,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=ragas_llm, embeddings=ragas_emb,
    raise_exceptions=False,
)
print(result_general)

if records_with_gt:
    ds_gt = Dataset.from_list([
        {
            "question": r["question"],
            "answer": r["answer"],
            "contexts": r["contexts"],
            "reference": r["ground_truth"],   # RAGAS 0.2 renamed ground_truth -> reference
        } for r in records_with_gt
    ])
    print(f"\nScoring Context Precision + Context Recall on {len(ds_gt)} records with reference ...")
    result_gt = evaluate(
        ds_gt,
        metrics=[ContextPrecision(), ContextRecall()],
        llm=ragas_llm, embeddings=ragas_emb,
        raise_exceptions=False,
    )
    print(result_gt)
else:
    result_gt = None
    print("\nNo records with ground_truth/reference; skipping Context Precision/Recall.")


## Cell 6 - Save report

In [ ]:
from datetime import datetime

def to_dict(scores):
    if scores is None: return None
    try: return scores.to_pandas().mean(numeric_only=True).to_dict()
    except Exception: return {k: float(v) for k, v in scores.items()}

report = {
    "n_queries": len(records),
    "n_with_reference": sum(1 for r in records if r["ground_truth"]),
    "judge_model": CONFIG["judge_model"],
    "embedding_model": CONFIG["embedding_model"],
    "top_k": CONFIG["top_k"],
    "mean_latency_s": round(sum(r["latency_s"] for r in records)/len(records), 2),
    "general_metrics": to_dict(result_general),
    "ground_truth_metrics": to_dict(result_gt) if result_gt else {},
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
}

out_path = EVAL_DIR / f"ragas_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Report saved: {out_path}")
print(json.dumps(report, indent=2, ensure_ascii=False))

# Also save per-query detail for thesis appendix
detail_path = EVAL_DIR / "ragas_detail.csv"
pd.DataFrame(records).to_csv(detail_path, index=False, encoding="utf-8")
print(f"Per-query CSV: {detail_path}")


---
### Interpreting the numbers

- **Faithfulness 0.8+** = answer rarely contradicts the context. Good grounding.
- **Answer Relevancy 0.7+** = answer addresses the question; <0.7 means rambling/off-topic.
- **Context Precision 0.7+** = retrieved chunks are mostly relevant (low noise).
- **Context Recall 0.7+** = retrieval pulls in everything needed for the ground-truth answer.

For your thesis, report all four with the per-query CSV as an appendix. Examiners value RAGAS because it's a peer-reviewed standard from the GitHub.com/explodinggradients team.
